In [ ]:
import numpy as np
import pandas as pd
import json
import spacy
import ast
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import GroupShuffleSplit, train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

path_data = '/content/drive/MyDrive/tesis_monica/afasia/data/'

Mounted at /content/drive


In [ ]:
df_aphbank = pd.read_csv(path_data + 'df_aphbank_resultados.csv', encoding='utf-8')
df_aphbank.loc[df_aphbank["name_chunk_audio_path"].str.contains("aphasiabank_en", na=False), "LLengWAB"] = 3

df_catalan = pd.read_csv(path_data + 'df_transcrip_audio_metrics.csv', encoding='utf-8')

#WAB_AQ_category -> aphasia type

df_catalan.loc[(pd.to_numeric(df_catalan['QA'])>= 0) & (pd.to_numeric(df_catalan['QA'])<=25), 'WAB_AQ_category'] = 'Very severe'
df_catalan.loc[(pd.to_numeric(df_catalan['QA'])> 25) & (pd.to_numeric(df_catalan['QA'])<=50), 'WAB_AQ_category'] = 'Severe'
df_catalan.loc[(pd.to_numeric(df_catalan['QA'])> 50) & (pd.to_numeric(df_catalan['QA'])<=75), 'WAB_AQ_category'] = 'Moderate'
df_catalan.loc[(pd.to_numeric(df_catalan['QA'])> 75) , 'WAB_AQ_category'] = 'Mild'
df_catalan.loc[np.isnan(pd.to_numeric(df_catalan['QA'])) , 'WAB_AQ_category'] = 'Unknown'

df_catalan['Grup'] = df_catalan['WAB_AQ_category']

# Eliminar la columna 'WAB_AQ_category' ya que no es necesaria
df_catalan.drop(columns=['WAB_AQ_category'], inplace=True)

df = pd.concat([df_aphbank, df_catalan], ignore_index=True)

In [ ]:
print(df_aphbank.shape[0])
print(df_catalan.shape[0])
print(df.shape[0])

77272
543
77815


In [ ]:
print(df_aphbank.dropna(subset=['QA']).shape[0])
print(df_catalan.dropna(subset=['QA']).shape[0])
print(df.dropna(subset=['QA']).shape[0])

70676
543
71219


In [ ]:
# Función para procesar la columna 'bert_embedding' y conservarla como vector (lista)
def process_bert_embedding(df):
    if "bert_embedding" not in df.columns:
        print("Advertencia: 'bert_embedding' no está en el DataFrame.")
        return df

    def convert_to_list(value):
        if isinstance(value, str):
            try:
                return json.loads(value)
            except json.JSONDecodeError:
                try:
                    return ast.literal_eval(value)
                except (ValueError, SyntaxError):
                    return None
        elif isinstance(value, (list, np.ndarray)):
            return list(value)
        return None

    df["bert_embedding"] = df["bert_embedding"].apply(convert_to_list)
    df = df[df["bert_embedding"].notnull()]
    return df

*  1 -> catalan
*  2 -> castellano
*  3 -> ingles

In [ ]:
# Procesar la columna 'bert_embedding'
df = process_bert_embedding(df)

# Eliminar filas sin QA (si fuera el caso)
df = df.dropna(subset=['QA'])

# Se asume que la columna LLengWAB indica:
#   1 -> catalán, 2 -> castellano, 3 -> inglés
# Filtrar según idioma
df_english = df[df["LLengWAB"] == 3].copy()
df_non_english = df[df["LLengWAB"].isin([1, 2])].copy()

print("Total casos:", df.shape[0])
print("Casos en inglés (para entrenamiento interno):", df_english.shape[0])
print("Casos en catalán/castellano (para test externo):", df_non_english.shape[0])

# Definir las columnas irrelevantes para el modelo
irrelevant_columns = ['Inicio', 'Fin', 'Transcrip_name', 'name_chunk_audio',
                      'name_chunk_audio_path', 'TipusAfàsia', 'Fluente/No Fluente']

Total casos: 71219
Casos en inglés (para entrenamiento interno): 69861
Casos en catalán/castellano (para test externo): 1358


In [ ]:
# --- SPLIT EN INGLÉS (70% train / 15% validación / 15% test interno) usando GroupShuffleSplit según CIP ---

# Seleccionar X e y del conjunto inglés
X_eng = df_english.drop(columns=irrelevant_columns + ['QA'])
X_eng = X_eng.select_dtypes(include=['number'])
y_eng = df_english['QA'].astype(float)
groups_eng = df_english["CIP"]

# Primer split: separar 70% train y 30% resto (para validación e interno test)
gss = GroupShuffleSplit(test_size=0.30, n_splits=1, random_state=42)
train_idx_eng, temp_idx_eng = next(gss.split(X_eng, y_eng, groups=groups_eng))
df_eng_train = df_english.iloc[train_idx_eng].copy()
df_eng_temp = df_english.iloc[temp_idx_eng].copy()

# Segundo split: dividir el 30% restante en 50/50 => ~15% validación y ~15% test interno
df_eng_val, df_eng_test = train_test_split(df_eng_temp, test_size=0.5, random_state=42, stratify=df_eng_temp['QA'])

print("\n=== Conjunto Inglés ===")
print("Tamaño del entrenamiento:", df_eng_train.shape)
print("Tamaño de validación:", df_eng_val.shape)
print("Tamaño del test interno:", df_eng_test.shape)



=== Conjunto Inglés ===
Tamaño del entrenamiento: (48536, 146)
Tamaño de validación: (10662, 146)
Tamaño del test interno: (10663, 146)


In [ ]:
# --- Conjunto de test externo: catalán y castellano ---
df_external_test = df_non_english.copy()
print("\nTamaño del test externo (catalán y castellano):", df_external_test.shape)

# Opcional: Seleccionar variables relevantes para el modelo (solo numéricas)
def preparar_datos(df_input):
    X = df_input.drop(columns=irrelevant_columns + ['QA'])
    X = X.select_dtypes(include=['number'])
    y = df_input['QA'].astype(float)
    return X, y

X_train, y_train = preparar_datos(df_eng_train)
X_val, y_val = preparar_datos(df_eng_val)
X_test_internal, y_test_internal = preparar_datos(df_eng_test)
X_test_external, y_test_external = preparar_datos(df_external_test)

# Mostrar algunas distribuciones y estadísticas
print("\n--- Distribución de 'Grup' en entrenamiento inglés ---")
print(df_eng_train["Grup"].value_counts())
print("\n--- Distribución de 'Grup' en validación inglés ---")
print(df_eng_val["Grup"].value_counts())
print("\n--- Distribución de 'Grup' en test interno inglés ---")
print(df_eng_test["Grup"].value_counts())
print("\n--- Distribución de 'Grup' en test externo (catalán/castellano) ---")
print(df_external_test["Grup"].value_counts())

print("\nPacientes en entrenamiento inglés:", df_eng_train["CIP"].unique())
print("Pacientes en validación inglés:", df_eng_val["CIP"].unique())
print("Pacientes en test interno inglés:", df_eng_test["CIP"].unique())
print("Pacientes en test externo:", df_external_test["CIP"].unique())

# Finalmente, se muestran las etiquetas QA de cada conjunto
print("\nEtiquetas en y_train:", set(y_train))
print("Etiquetas en y_val:", set(y_val))
print("Etiquetas en y_test_internal:", set(y_test_internal))
print("Etiquetas en y_test_external:", set(y_test_external))


Tamaño del test externo (catalán y castellano): (1358, 146)

--- Distribución de 'Grup' en entrenamiento inglés ---
Grup
Mild           23339
Moderate       19790
Severe          4140
Very severe     1267
Name: count, dtype: int64

--- Distribución de 'Grup' en validación inglés ---
Grup
Mild           6017
Moderate       3486
Severe          951
Very severe     208
Name: count, dtype: int64

--- Distribución de 'Grup' en test interno inglés ---
Grup
Mild           6018
Moderate       3487
Severe          951
Very severe     207
Name: count, dtype: int64

--- Distribución de 'Grup' en test externo (catalán/castellano) ---
Grup
Mild       574
Unknown    461
Severe     323
Name: count, dtype: int64

Pacientes en entrenamiento inglés: ['BU02a' 'elman15a' 'whiteside06a' 'williamson15a' 'williamson05a'
 'tcu08a' 'whiteside02a' 'thompson13a' 'scale15d' 'tcu10b' 'BU12a'
 'scale07a' 'BU10a' 'elman07a' 'adler21a' 'whiteside10a' 'adler20a'
 'adler10a' 'tucson19a' 'scale20a' 'adler22a' 'scale11a

In [ ]:
df_eng_train.to_csv(path_data + 'df_eng_train.csv', index=False, encoding='utf-8')
df_eng_val.to_csv(path_data + 'df_eng_val.csv', index=False, encoding='utf-8')
df_eng_test.to_csv(path_data + 'df_eng_test.csv', index=False, encoding='utf-8')
df_external_test.to_csv(path_data + 'df_external_test.csv', index=False, encoding='utf-8')